In [20]:
import os
import transformers
from helper_functions import *
import json





batch_path = "eval_p3"
transform_lct ="/work/eauten2s/ec_criteria_struct/lct"

# Model
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"

# N Shots
n_shot = 5 # liefert genau die Anzahl Beispiele (study, label)

# Input/ Output
study_path = f"{transform_lct}/input/lct_txt_half/"

# Model
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"
n_shot=2
# Input/ Output
study_path = f"{transform_lct}/input/lct_txt_half/"
output_path = f"{transform_lct}/evaluate/{batch_path}/model_output/{model_name}_{n_shot}_shot/output/"
os.makedirs(output_path, exist_ok=True)


anfang = 0
ende = 300
study_files = os.listdir(study_path)[anfang:ende]


# Load Model Description
model_desc = read_text_file(f"{transform_lct}/input/prompt/prompt_p3.txt")

def read_matching_p3_files(study_folder, n_shot):
    study_filenames = []
    study_contents = []
    label_filenames = []
    label_contents = []

    loaded_files = 0
    for file_name in os.listdir(study_folder):
        if file_name.endswith(".txt"):
            study_filenames.append(file_name)
            study_file_path = os.path.join(study_folder, file_name)
            with open(study_file_path, 'r', encoding='utf-8') as file:
                study_contents.append(file.read())

            label_file_name = file_name.replace(".txt", "_p3.json")
            label_file_path = os.path.join(study_folder, label_file_name)
            print(label_file_path)
            if os.path.exists(label_file_path):
                label_filenames.append(label_file_name)
                with open(label_file_path, 'r', encoding='utf-8') as file:
                    label_contents.append(file.read())
            else:
                label_filenames.append(None)
                label_contents.append(None)
            loaded_files += 1
            if loaded_files == n_shot * 2:
                break

    return study_filenames, study_contents, label_filenames, label_contents


# Load n-shot Data
study_folder = f"{transform_lct}/input/n_shot_files_p3"

study_filenames, study_contents, label_filenames, label_contents = read_matching_p3_files(study_folder, n_shot)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []

command = "Bring the following eligibility criterias in Json format with logical operators and extract entitys:"

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})





first_call = True

for file in study_files:

    print(file)
    test_file = read_text_file(study_path+file)
    print(test_file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}




In [21]:
messages

### Test n-shot Input

In [5]:

import os
import transformers
from helper_functions import *

batch_path = "eval_p1"
n_prompt = 7
n_shot = 5
temp=0.6
temp_str = f"_temp_{str(temp).split('.')[1]}"
model_id =  "meta-llama/Meta-Llama-3-70B-Instruct"
model_name = "Llama-3-70B-Instruct"
transform_lct ="/work/eauten2s/ec_criteria_struct/lct"
model_desc = read_text_file(f"{transform_lct}/input/prompt/p{n_prompt}.txt")

study_path = f"{transform_lct}/input/lct_txt/"
output_path = f"{transform_lct}/evaluate/{batch_path}/model_output/{model_name}_{n_shot}_shot_prompt_{n_prompt}{temp_str}_cot/output/"
os.makedirs(output_path, exist_ok=True)


study_files = os.listdir(study_path)[:100]

shot_list = [
    "NCT03865433.txt",
    "NCT03860324.txt",
    "NCT03860233.txt",
    "NCT03923231.txt",
    "NCT03930121.txt"
]

one_shot_list = ["NCT03923231.txt"]

# Load n-shot Data
study_folder = f"{transform_lct}/input/lct_txt/"
label_folder = f'{transform_lct}/input/lct_p1'
study_filenames, study_contents, label_filenames, label_contents = read_matching_txt_files(study_folder, label_folder, shot_list)
studies = dict(zip(study_filenames, study_contents))
labels = dict(zip(label_filenames, label_contents))
messages = []




cot = "Let's think through this carefully, step by step."

command = "Insert the logical operators [AND], [OR], [NOT] into the following eligibility criteria and return the text in full without deleting/replacing anything. Do not say anything else." 

messages.append({"role": "system", "content": f"{model_desc}"})

for i in range(n_shot):
    messages.append({"role": "user", "content": f"{command} {studies[study_filenames[i]]}"})
    messages.append({"role": "assistant", "content": labels[label_filenames[i]]})


first_call = True

for file in study_files:
    file_name = file.split(".")[0]
    print("File:", file_name, "\n")
    
    test_file = read_text_file(study_path+file)

    if first_call:
        messages.append({"role": "user", "content": f"{command} {test_file}"})
        first_call = False
    else:
        messages[-1] = {"role": "user", "content": f"{command} {test_file}"}


Das File: /work/eauten2s/ec_criteria_struct/lct/input/prompt/p7.txt wurde erfolgreich geladen.
File: NCT03860012 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860012.txt wurde erfolgreich geladen.
File: NCT03860025 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860025.txt wurde erfolgreich geladen.
File: NCT03860038 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860038.txt wurde erfolgreich geladen.
File: NCT03860064 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860064.txt wurde erfolgreich geladen.
File: NCT03860090 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860090.txt wurde erfolgreich geladen.
File: NCT03860103 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860103.txt wurde erfolgreich geladen.
File: NCT03860116 

Das File: /work/eauten2s/ec_criteria_struct/lct/input/lct_txt/NCT03860116.txt wurde erfolgreich geladen.
File: NCT03860142 

Das File: 

In [6]:
messages

[{'role': 'system',
  'content': 'Your role is  a professional text editor specialized in accurately and precisely editing texts. Your task is to review the given eligibility criteria and correctly insert the logical operators [AND], [OR], [NOT] into the text. \nMake sure not to add any additional remarks or comments. \n\nCreate an edited version of the following inclusion and exclusion criteria that incorporates the logical operators [AND], [OR], [NOT]. Apply these rules:\n[OR]:\nAlways write an [OR] before an "or" in the text.\nAlways write an [OR] after a comma when it separates alternative options or conditions.\n[AND]:\nAlways write an [AND] before "with" when it connects related conditions or requirements that must all be satisfied.\nWrite an [AND] before "and" when it joins two or more conditions or criteria that must all be met for eligibility.\n[NOT]:\nAlways write a [NOT] before "no", "not", "unable", "none", "inadequate", "unqualified", or "inability" when these words indica